In [1]:
#Capstone Project - Cease and Desist Doc Processing
from google.colab import drive
drive.mount('/content/drive' , force_remount = True)

Mounted at /content/drive


In [41]:
import os
import sys
folder_path = "/content/drive/MyDrive/pdf"


In [ ]:
#these are already installed


In [ ]:
#!pip install -U \
#langchain==0.2.16 \
#langchain-core==0.2.38 \
#langchain-community==0.2.16 \
#langchain-groq

In [103]:
import os, json, re, sqlite3
from typing import TypedDict, Dict, Any

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader


In [43]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

In [45]:
##llm model
from langchain_groq import ChatGroq
MODEL_NAME = "llama-3.1-8b-instant"
#MODEL_NAME = "llama3-70b-8192"

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name=MODEL_NAME,
    temperature=0
)

In [131]:
import os
import pytesseract
from pdf2image import convert_from_path
import time
from langchain_community.document_loaders import PyPDFLoader

# OCR for scanned PDFs
def run_ocr(file_path):

    print(f"OCR running for: {file_path}")

    images = convert_from_path(file_path)
    text = ""

    for img in images:
        text += pytesseract.image_to_string(img) + "\n"

    return text


#  PDF LOADER
def load_all_pdfs(folder_path):

    documents = []

    print("\n📂 Reading folder:", folder_path)

    # Step 1: Show ALL files
    all_files = os.listdir(folder_path)
    print("\n📁 All files in folder:")
    for f in all_files:
        print("   ", f)

    # Step 2: Process PDFs
    for file in all_files:

        #print("\n Found file:", file)

        if file.lower().endswith(".pdf"):   #  handle .PDF also

            file_path = os.path.join(folder_path, file)

            print(f"Loading: {file}")

            loader = PyPDFLoader(file_path)
            pages = loader.load()

            text = "\n".join([p.page_content for p in pages])

            # OCR fallback
            if len(text.strip()) < 50:
                print("Low text → using OCR")
                text = run_ocr(file_path)

            documents.append({
                "file": file,
                "text": text
            })

    print(f"\n Loaded {len(documents)} documents")

    return documents


# RUN
docs = load_all_pdfs(folder_path)


# Print all loaded docs
print("\n FINAL DOCUMENT LIST:\n")

for doc in docs:
    print("📄", doc["file"])
    print("Text preview:", doc["text"][:200])
    print("-" * 60)


📂 Reading folder: /content/drive/MyDrive/pdf

📁 All files in folder:
    LOA2.pdf
    LOA3.pdf
    LOA4.pdf
    LOA5.pdf
    LOA6.pdf
    LOA7.pdf
    LOA8.pdf
    LOA9.pdf
    LoA1.pdf
    bw_doc_1.pdf
    bw_doc_2.pdf
    bw_doc_3.pdf
    bw_doc_4.pdf
    bw_doc_5.pdf
    notice_1.pdf
    notice_2.pdf
    notice_3.pdf
    notice_4.pdf
    notice_5.pdf
Loading: LOA2.pdf
Loading: LOA3.pdf
Loading: LOA4.pdf
Loading: LOA5.pdf
Loading: LOA6.pdf
Loading: LOA7.pdf
Loading: LOA8.pdf
Loading: LOA9.pdf
Loading: LoA1.pdf
Loading: bw_doc_1.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_1.pdf
Loading: bw_doc_2.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_2.pdf
Loading: bw_doc_3.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_3.pdf
Loading: bw_doc_4.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_4.pdf
Loading: bw_doc_5.pdf
Low text → using OCR
OCR running for: /content/drive/MyDriv

In [132]:
import time

LAST_CALL = 0
MIN_DELAY = 2.5   # increase to 3–4 if still failing

def throttle():
    global LAST_CALL
    now = time.time()

    gap = now - LAST_CALL
    if gap < MIN_DELAY:
        time.sleep(MIN_DELAY - gap)

    LAST_CALL = time.time()

In [133]:
# To aviod any error for limit rate
def safe_llm(prompt, retries=3):

    for i in range(retries):
        try:
            throttle()
            return llm.invoke(prompt).content

        except Exception as e:

            if "429" in str(e):
                wait = 4 + i * 2
                print(f"⏳ Rate limit → retry in {wait}s")
                time.sleep(wait)
            else:
                raise e

    return "Error"

In [134]:
# Confidence label helper
def confidence_label(score):

    if score >= 0.85:
        return "Very High"
    elif score >= 0.7:
        return "Medium"
    elif score >= 0.5:
        return "Low"
    else:
        return "Very Low"

# print classification confidence function and quality
def print_confidence(cls_conf, extraction_conf):

    cls_label = confidence_label(cls_conf)
    ext_label = confidence_label(extraction_conf)

    print("\nCONFIDENCE SUMMARY")
    print("-" * 40)

    print(f"Classification Confidence : {cls_label} ({round(cls_conf,2)})")
    print(f"Extraction Quality        : {ext_label} ({round(extraction_conf,2)})")

    if extraction_conf < 0.7:
        print("Recommendation: Human review suggested")
    else:
        print("Recommendation: Safe to auto-approve")

    print("-" * 40)

In [136]:
## classificaiton tool for LOA, notice and Business doc
@tool
def classify_loa(text: str) -> str:
    """Check if doc is LOA. Returns Yes/No + Confidence."""
    return safe_llm(f"""
    Is this a Letter of Authorization (LOA)?
    Answer: Yes or No
    Confidence: 0-1
    Reason:

    {text[:800]}
    """)

@tool
def classify_notice(text: str) -> str:
    """Check if doc is Notice."""
    return safe_llm(f"""
    Is this a legal notice?
    Answer: Yes or No
    Confidence: 0-1
    Reason:

    {text[:800]}
    """)

@tool
def classify_business(text: str) -> str:
    """Check if doc is Business document."""
    return safe_llm(f"""
    Is this a business document?
    Answer: Yes or No
    Confidence: 0-1
    Reason:

    {text[:800]}
    """)

In [137]:
##Extraction
@tool
def extract_loa(text: str) -> str:
    """Extract LOA details (flexible)."""
    return safe_llm(f"""
    Extract key LOA details (natural text).
    Focus: client, agent, authority, dates.

    {text[:800]}
    """)

@tool
def extract_notice(text: str) -> str:
    """Extract Notice details."""
    return safe_llm(f"""
    Extract notice info: sender, recipient, issue, demand.

    {text[:800]}
    """)

@tool
def extract_business(text: str) -> str:
    """Extract Business doc details."""
    return safe_llm(f"""
    Extract business info: parties, agreement, terms.

    {text[:800]}
    """)

In [138]:
def validate_with_keys(text, doc_type):

    return safe_llm(f"""
    Analyze this {doc_type} document.

    Identify:
    - missing fields
    - unclear or noisy text
    - incomplete information

    STRICTLY return:

    Issues:
    - bullet points

    Confidence: (a number between 0 and 1, like 0.65 or 0.9)

    Document:
    {text[:1500]}
    """)

def get_confidence(text):

    # Try to find "Confidence: 0.X"
    match = re.search(r"Confidence\s*:\s*(\d*\.?\d+)", text, re.IGNORECASE)

    if match:
        return float(match.group(1))

    print("Confidence not found → defaulting to 0.5")
    return 0.5

In [139]:
##CLASSIFIERS and DECISION for faster result
def run_classifiers(text):

    loa = classify_loa.invoke(text)
    time.sleep(1)

    notice = classify_notice.invoke(text)
    time.sleep(1)

    business = classify_business.invoke(text)

    return loa, notice, business

def extract_conf(text):
    try:
        return float(text.split("Confidence:")[-1].strip().split()[0])
    except:
        return 0.0

def choose_doc_type(loa, notice, business):
    scores = {
        "LOA": extract_conf(loa),
        "NOTICE": extract_conf(notice),
        "BUSINESS": extract_conf(business)
    }
    best = max(scores, key=scores.get)
    return best, scores[best]

In [140]:
##Sqlite and memory
import concurrent.futures
conn = sqlite3.connect("documents.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS documents (
id INTEGER PRIMARY KEY AUTOINCREMENT,
file TEXT,
type TEXT,
confidence REAL,
output TEXT
)
""")
conn.commit()

memory_store = []

def save_result(doc, doc_type, confidence, output):
    cur.execute(
        "INSERT INTO documents (file,type,confidence,output) VALUES (?,?,?,?)",
        (doc["file"], doc_type, confidence, output)
    )
    conn.commit()

    memory_store.append({
        "file": doc["file"],
        "type": doc_type,
        "confidence": confidence,
        "output": output
    })

In [141]:
def human_review(doc, extracted, text, doc_type):

    current_output = extracted

    while True:

        print("\n HUMAN REVIEW REQUIRED")
        print("\n Document:", doc["file"])

        print("\n Current Output:\n")
        print(current_output)

        # GET AI FEEDBACK HERE
        validation = validate_with_keys(current_output, doc_type)
        confidence = round(get_confidence(validation), 2)

        print("\n AI Review:\n")
        print(validation)

        print(f"\n AI Confidence: {confidence}")

        print("\n Choose action:")
        print("1 → Accept")
        print("2 → Reject & Correct")

        choice = input("Enter 1 or 2: ").strip()

        # ACCEPT
        if choice == "1":
            print("\n Accepted by human")
            return current_output

        # CORRECT
        elif choice == "2":

            print("\n Enter corrected version:")
            corrected = input()

            if not corrected.strip():
                print(" Empty correction — try again")
                continue

            current_output = corrected  # update

            # RE-VALIDATE AGAIN for confidence is greater than .7
            validation = validate_with_keys(current_output, doc_type)
            confidence = round(get_confidence(validation), 2)

            print("\n Re-validation result:")
            print(validation)

            print(f"\n New Confidence: {confidence}")

            if confidence > 0.7:
                print("\n Correction accepted")
                return current_output
            else:
                print("\n Still low confidence — please improve")

        else:
            print(" Invalid choice — try again")

In [143]:
def process_document(doc):

    print("\n" + "="*80)
    print(f"📄 PROCESSING FILE: {doc['file']}")


    text = doc["text"]

    # ---------------- CLASSIFICATION ---------------- #
    loa, notice, business = run_classifiers(text)

    # ADD small delay after classifiers (IMPORTANT)
    import time
    time.sleep(1.5)

    doc_type, cls_conf = choose_doc_type(loa, notice, business)

    print(f"\n Document Type: {doc_type}")

    # ---------------- EXTRACTION ---------------- #
    if doc_type == "LOA":
        extracted = extract_loa.invoke(text)

    elif doc_type == "NOTICE":
        extracted = extract_notice.invoke(text)

    else:
        extracted = extract_business.invoke(text)

    # ADD delay after extraction
    time.sleep(1.5)

    print("\n Extraction ready")
    print("\n Selected Extraction:\n")
    print(extracted)

    # ---------------- VALIDATION ---------------- #

    # OPTIONAL OPTIMIZATION (reduces 1 LLM call)
    if cls_conf > 0.9:
        print("\n⚡ High classification confidence → skipping validation")
        confidence = 0.9
    else:
        validation = validate_with_keys(extracted, doc_type)

        # ADD delay after validation
        time.sleep(1.5)

        confidence = round(get_confidence(validation), 2)

    # ---------------- CONFIDENCE PRINT ---------------- #

    cls_label = confidence_label(cls_conf)
    ext_label = confidence_label(confidence)

    print("\n CONFIDENCE SUMMARY")
    print("-" * 40)
    print(f"Classification Confidence : {cls_label} ({round(cls_conf,2)})")
    print(f"Extraction Quality        : {ext_label} ({confidence})")

    if confidence < 0.85:
        print("Recommendation: Human review suggested")
    else:
        print("Recommendation: Safe to auto-approve")
    print("-" * 40)

    # ---------------- HITL ---------------- #

    if confidence < 0.85:
        final = human_review(doc, extracted, text, doc_type)
    else:
        print("\n AUTO APPROVED")
        final = extracted

    # ---------------- SAVE ---------------- #
    save_result(doc, doc_type, confidence, final)

    print("\n Saved")
    return final

In [144]:
#folder_path = "/content/drive/MyDrive/pdf"
docs = load_all_pdfs(folder_path)

for doc in docs:
    process_document(doc)
    time.sleep(3)


📂 Reading folder: /content/drive/MyDrive/pdf

📁 All files in folder:
    LOA2.pdf
    LOA3.pdf
    LOA4.pdf
    LOA5.pdf
    LOA6.pdf
    LOA7.pdf
    LOA8.pdf
    LOA9.pdf
    LoA1.pdf
    bw_doc_1.pdf
    bw_doc_2.pdf
    bw_doc_3.pdf
    bw_doc_4.pdf
    bw_doc_5.pdf
    notice_1.pdf
    notice_2.pdf
    notice_3.pdf
    notice_4.pdf
    notice_5.pdf
Loading: LOA2.pdf
Loading: LOA3.pdf
Loading: LOA4.pdf
Loading: LOA5.pdf
Loading: LOA6.pdf
Loading: LOA7.pdf
Loading: LOA8.pdf
Loading: LOA9.pdf
Loading: LoA1.pdf
Loading: bw_doc_1.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_1.pdf
Loading: bw_doc_2.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_2.pdf
Loading: bw_doc_3.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_3.pdf
Loading: bw_doc_4.pdf
Low text → using OCR
OCR running for: /content/drive/MyDrive/pdf/bw_doc_4.pdf
Loading: bw_doc_5.pdf
Low text → using OCR
OCR running for: /content/drive/MyDriv